# Pastel Tres Leches - Place & Route (interactivo)

Capsula 7. Ya tenemos el **netlist** (Polvorones). El PnR lo volvio **geometria**:
coloco cada compuerta y las conecto con capas de metal. Aqui **vemos** ese layout
-- el `semaforo` real sobre SKY130, generado con OpenROAD -- y jugamos con el.

> El PnR pesado corre en el servidor (OpenROAD); este notebook es el **visor**:
> lee el `.def` (el layout en texto) y lo dibuja con **puro Python**, sin herramientas
> EDA. El layout va embebido, asi que no hay que instalar ni clonar nada.

Corre las celdas en orden.

## 1 - Preparar (el layout va embebido)

In [ ]:
import os, re
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from ipywidgets import interact, Dropdown, Checkbox

DEF_TEXT = r'''VERSION 5.8 ;
DIVIDERCHAR "/" ;
BUSBITCHARS "[]" ;
DESIGN semaforo ;
UNITS DISTANCE MICRONS 1000 ;
DIEAREA ( 0 0 ) ( 41525 41525 ) ;
ROW ROW_0 unithd 3220 5440 N DO 76 BY 1 STEP 460 0 ;
ROW ROW_1 unithd 3220 8160 FS DO 76 BY 1 STEP 460 0 ;
ROW ROW_2 unithd 3220 10880 N DO 76 BY 1 STEP 460 0 ;
ROW ROW_3 unithd 3220 13600 FS DO 76 BY 1 STEP 460 0 ;
ROW ROW_4 unithd 3220 16320 N DO 76 BY 1 STEP 460 0 ;
ROW ROW_5 unithd 3220 19040 FS DO 76 BY 1 STEP 460 0 ;
ROW ROW_6 unithd 3220 21760 N DO 76 BY 1 STEP 460 0 ;
ROW ROW_7 unithd 3220 24480 FS DO 76 BY 1 STEP 460 0 ;
ROW ROW_8 unithd 3220 27200 N DO 76 BY 1 STEP 460 0 ;
ROW ROW_9 unithd 3220 29920 FS DO 76 BY 1 STEP 460 0 ;
ROW ROW_10 unithd 3220 32640 N DO 76 BY 1 STEP 460 0 ;
ROW ROW_11 unithd 3220 35360 FS DO 76 BY 1 STEP 460 0 ;
TRACKS X 230 DO 90 STEP 460 LAYER li1 ;
TRACKS Y 170 DO 122 STEP 340 LAYER li1 ;
TRACKS X 170 DO 122 STEP 340 LAYER met1 ;
TRACKS Y 170 DO 122 STEP 340 LAYER met1 ;
TRACKS X 230 DO 90 STEP 460 LAYER met2 ;
TRACKS Y 230 DO 90 STEP 460 LAYER met2 ;
TRACKS X 340 DO 61 STEP 680 LAYER met3 ;
TRACKS Y 340 DO 61 STEP 680 LAYER met3 ;
TRACKS X 460 DO 45 STEP 920 LAYER met4 ;
TRACKS Y 460 DO 45 STEP 920 LAYER met4 ;
TRACKS X 1700 DO 12 STEP 3400 LAYER met5 ;
TRACKS Y 1700 DO 12 STEP 3400 LAYER met5 ;
GCELLGRID X 0 DO 6 STEP 6900 ;
GCELLGRID Y 0 DO 6 STEP 6900 ;
NONDEFAULTRULES 1 ;
    - CTS_NDR_0
      + LAYER li1 WIDTH 170 SPACING 580
      + LAYER met1 WIDTH 140 SPACING 400
      + LAYER met2 WIDTH 140 SPACING 640
      + LAYER met3 WIDTH 300 SPACING 760
      + LAYER met4 WIDTH 300 SPACING 1240
      + LAYER met5 WIDTH 1600 SPACING 3600
    ;
END NONDEFAULTRULES
COMPONENTS 80 ;
    - PHY_EDGE_ROW_0_Left_12 sky130_fd_sc_hd__decap_3 + SOURCE DIST + FIXED ( 3220 5440 ) N ;
    - PHY_EDGE_ROW_0_Right_0 sky130_fd_sc_hd__decap_3 + SOURCE DIST + FIXED ( 36800 5440 ) FN ;
    - PHY_EDGE_ROW_10_Left_22 sky130_fd_sc_hd__decap_3 + SOURCE DIST + FIXED ( 3220 32640 ) N ;
    - PHY_EDGE_ROW_10_Right_10 sky130_fd_sc_hd__decap_3 + SOURCE DIST + FIXED ( 36800 32640 ) FN ;
    - PHY_EDGE_ROW_11_Left_23 sky130_fd_sc_hd__decap_3 + SOURCE DIST + FIXED ( 3220 35360 ) FS ;
    - PHY_EDGE_ROW_11_Right_11 sky130_fd_sc_hd__decap_3 + SOURCE DIST + FIXED ( 36800 35360 ) S ;
    - PHY_EDGE_ROW_1_Left_13 sky130_fd_sc_hd__decap_3 + SOURCE DIST + FIXED ( 3220 8160 ) FS ;
    - PHY_EDGE_ROW_1_Right_1 sky130_fd_sc_hd__decap_3 + SOURCE DIST + FIXED ( 36800 8160 ) S ;
    - PHY_EDGE_ROW_2_Left_14 sky130_fd_sc_hd__decap_3 + SOURCE DIST + FIXED ( 3220 10880 ) N ;
    - PHY_EDGE_ROW_2_Right_2 sky130_fd_sc_hd__decap_3 + SOURCE DIST + FIXED ( 36800 10880 ) FN ;
    - PHY_EDGE_ROW_3_Left_15 sky130_fd_sc_hd__decap_3 + SOURCE DIST + FIXED ( 3220 13600 ) FS ;
    - PHY_EDGE_ROW_3_Right_3 sky130_fd_sc_hd__decap_3 + SOURCE DIST + FIXED ( 36800 13600 ) S ;
    - PHY_EDGE_ROW_4_Left_16 sky130_fd_sc_hd__decap_3 + SOURCE DIST + FIXED ( 3220 16320 ) N ;
    - PHY_EDGE_ROW_4_Right_4 sky130_fd_sc_hd__decap_3 + SOURCE DIST + FIXED ( 36800 16320 ) FN ;
    - PHY_EDGE_ROW_5_Left_17 sky130_fd_sc_hd__decap_3 + SOURCE DIST + FIXED ( 3220 19040 ) FS ;
    - PHY_EDGE_ROW_5_Right_5 sky130_fd_sc_hd__decap_3 + SOURCE DIST + FIXED ( 36800 19040 ) S ;
    - PHY_EDGE_ROW_6_Left_18 sky130_fd_sc_hd__decap_3 + SOURCE DIST + FIXED ( 3220 21760 ) N ;
    - PHY_EDGE_ROW_6_Right_6 sky130_fd_sc_hd__decap_3 + SOURCE DIST + FIXED ( 36800 21760 ) FN ;
    - PHY_EDGE_ROW_7_Left_19 sky130_fd_sc_hd__decap_3 + SOURCE DIST + FIXED ( 3220 24480 ) FS ;
    - PHY_EDGE_ROW_7_Right_7 sky130_fd_sc_hd__decap_3 + SOURCE DIST + FIXED ( 36800 24480 ) S ;
    - PHY_EDGE_ROW_8_Left_20 sky130_fd_sc_hd__decap_3 + SOURCE DIST + FIXED ( 3220 27200 ) N ;
    - PHY_EDGE_ROW_8_Right_8 sky130_fd_sc_hd__decap_3 + SOURCE DIST + FIXED ( 36800 27200 ) FN ;
    - PHY_EDGE_ROW_9_Left_21 sky130_fd_sc_hd__decap_3 + SOURCE DIST + FIXED ( 3220 29920 ) FS ;
    - PHY_EDGE_ROW_9_Right_9 sky130_fd_sc_hd__decap_3 + SOURCE DIST + FIXED ( 36800 29920 ) S ;
    - TAP_TAPCELL_ROW_0_24 sky130_fd_sc_hd__tapvpwrvgnd_1 + SOURCE DIST + FIXED ( 17020 5440 ) N ;
    - TAP_TAPCELL_ROW_0_25 sky130_fd_sc_hd__tapvpwrvgnd_1 + SOURCE DIST + FIXED ( 30820 5440 ) N ;
    - TAP_TAPCELL_ROW_10_35 sky130_fd_sc_hd__tapvpwrvgnd_1 + SOURCE DIST + FIXED ( 17020 32640 ) N ;
    - TAP_TAPCELL_ROW_11_36 sky130_fd_sc_hd__tapvpwrvgnd_1 + SOURCE DIST + FIXED ( 17020 35360 ) FS ;
    - TAP_TAPCELL_ROW_11_37 sky130_fd_sc_hd__tapvpwrvgnd_1 + SOURCE DIST + FIXED ( 30820 35360 ) FS ;
    - TAP_TAPCELL_ROW_1_26 sky130_fd_sc_hd__tapvpwrvgnd_1 + SOURCE DIST + FIXED ( 30820 8160 ) FS ;
    - TAP_TAPCELL_ROW_2_27 sky130_fd_sc_hd__tapvpwrvgnd_1 + SOURCE DIST + FIXED ( 17020 10880 ) N ;
    - TAP_TAPCELL_ROW_3_28 sky130_fd_sc_hd__tapvpwrvgnd_1 + SOURCE DIST + FIXED ( 30820 13600 ) FS ;
    - TAP_TAPCELL_ROW_4_29 sky130_fd_sc_hd__tapvpwrvgnd_1 + SOURCE DIST + FIXED ( 17020 16320 ) N ;
    - TAP_TAPCELL_ROW_5_30 sky130_fd_sc_hd__tapvpwrvgnd_1 + SOURCE DIST + FIXED ( 30820 19040 ) FS ;
    - TAP_TAPCELL_ROW_6_31 sky130_fd_sc_hd__tapvpwrvgnd_1 + SOURCE DIST + FIXED ( 17020 21760 ) N ;
    - TAP_TAPCELL_ROW_7_32 sky130_fd_sc_hd__tapvpwrvgnd_1 + SOURCE DIST + FIXED ( 30820 24480 ) FS ;
    - TAP_TAPCELL_ROW_8_33 sky130_fd_sc_hd__tapvpwrvgnd_1 + SOURCE DIST + FIXED ( 17020 27200 ) N ;
    - TAP_TAPCELL_ROW_9_34 sky130_fd_sc_hd__tapvpwrvgnd_1 + SOURCE DIST + FIXED ( 30820 29920 ) FS ;
    - _19_ sky130_fd_sc_hd__and3_1 + PLACED ( 29440 16320 ) N ;
    - _20_ sky130_fd_sc_hd__nand2_1 + PLACED ( 27600 24480 ) FS ;
    - _21_ sky130_fd_sc_hd__and3_1 + PLACED ( 20240 29920 ) FS ;
    - _22_ sky130_fd_sc_hd__nand2_1 + PLACED ( 12880 35360 ) FS ;
    - _23_ sky130_fd_sc_hd__nand3_1 + PLACED ( 17480 29920 ) FS ;
    - _24_ sky130_fd_sc_hd__xnor2_1 + PLACED ( 11040 29920 ) FS ;
    - _25_ sky130_fd_sc_hd__xor2_1 + PLACED ( 7820 29920 ) FS ;
    - _26_ sky130_fd_sc_hd__xnor2_1 + PLACED ( 22540 29920 ) FS ;
    - _27_ sky130_fd_sc_hd__xor2_1 + PLACED ( 26680 29920 ) FS ;
    - _28_ sky130_fd_sc_hd__nor2_1 + PLACED ( 26220 19040 ) FS ;
    - _29_ sky130_fd_sc_hd__nor4_1 + PLACED ( 17480 27200 ) N ;
    - _30_ sky130_fd_sc_hd__lpflow_isobufsrc_1 + PLACED ( 22080 16320 ) N ;
    - _31_ sky130_fd_sc_hd__nand4_1 + PLACED ( 17480 16320 ) N ;
    - _32_ sky130_fd_sc_hd__nor2_1 + PLACED ( 19780 19040 ) FS ;
    - _33_ sky130_fd_sc_hd__nand4_1 + PLACED ( 19780 16320 ) N ;
    - _34_ sky130_fd_sc_hd__nand2_1 + PLACED ( 15180 13600 ) FS ;
    - _35_ sky130_fd_sc_hd__a21oi_1 + PLACED ( 27600 16320 ) N ;
    - _36_ sky130_fd_sc_hd__nor3_1 + PLACED ( 25760 16320 ) N ;
    - _37_ sky130_fd_sc_hd__xnor2_1 + PLACED ( 26680 13600 ) FS ;
    - _38_ sky130_fd_sc_hd__nor2_1 + PLACED ( 25300 13600 ) FS ;
    - _39_ sky130_fd_sc_hd__nor2b_1 + PLACED ( 20700 10880 ) N ;
    - _40_ sky130_fd_sc_hd__mux2_1 + PLACED ( 8740 13600 ) FS ;
    - _41_ sky130_fd_sc_hd__nand2_1 + PLACED ( 15640 10880 ) N ;
    - _42_ sky130_fd_sc_hd__nand2_1 + PLACED ( 12880 13600 ) FS ;
    - _43_ sky130_fd_sc_hd__nor2_1 + PLACED ( 8280 16320 ) N ;
    - _44_ sky130_fd_sc_hd__a22o_1 + PLACED ( 9660 16320 ) N ;
    - _45_ sky130_fd_sc_hd__xnor2_1 + PLACED ( 14260 29920 ) FS ;
    - _46_ sky130_fd_sc_hd__dfstp_1 + PLACED ( 5060 8160 ) FS ;
    - _47_ sky130_fd_sc_hd__dfrtp_1 + PLACED ( 4600 10880 ) N ;
    - _48_ sky130_fd_sc_hd__dfrtp_1 + PLACED ( 8740 19040 ) FS ;
    - _49_ sky130_fd_sc_hd__dfrtp_1 + PLACED ( 21620 8160 ) FS ;
    - _50_ sky130_fd_sc_hd__dfrtp_1 + PLACED ( 24840 10880 ) N ;
    - _51_ sky130_fd_sc_hd__dfrtp_1 + PLACED ( 24840 21760 ) N ;
    - _52_ sky130_fd_sc_hd__dfrtp_1 + PLACED ( 24840 32640 ) N ;
    - _53_ sky130_fd_sc_hd__dfrtp_1 + PLACED ( 24840 27200 ) N ;
    - _54_ sky130_fd_sc_hd__dfrtp_1 + PLACED ( 7820 32640 ) N ;
    - _55_ sky130_fd_sc_hd__dfrtp_1 + PLACED ( 8280 24480 ) FS ;
    - _56_ sky130_fd_sc_hd__dfrtp_1 + PLACED ( 7820 27200 ) N ;
    - clkbuf_0_clk sky130_fd_sc_hd__clkbuf_4 + SOURCE TIMING + PLACED ( 14260 21760 ) N ;
    - clkbuf_1_0__f_clk sky130_fd_sc_hd__clkbuf_4 + SOURCE TIMING + PLACED ( 17480 10880 ) N ;
    - clkbuf_1_1__f_clk sky130_fd_sc_hd__clkbuf_4 + SOURCE TIMING + PLACED ( 17480 32640 ) N ;
    - clkload0 sky130_fd_sc_hd__clkbuf_4 + SOURCE TIMING + PLACED ( 14720 8160 ) FS ;
END COMPONENTS
PINS 5 ;
    - clk + NET clk + DIRECTION INPUT + USE SIGNAL
      + PORT
        + LAYER met3 ( -400 -150 ) ( 400 150 )
        + PLACED ( 400 18020 ) N ;
    - green + NET state[0] + DIRECTION OUTPUT + USE SIGNAL
      + PORT
        + LAYER met3 ( -400 -150 ) ( 400 150 )
        + PLACED ( 400 23460 ) N ;
    - red + NET red + DIRECTION OUTPUT + USE SIGNAL
      + PORT
        + LAYER met3 ( -400 -150 ) ( 400 150 )
        + PLACED ( 400 19380 ) N ;
    - rst_n + NET rst_n + DIRECTION INPUT + USE SIGNAL
      + PORT
        + LAYER met3 ( -400 -150 ) ( 400 150 )
        + PLACED ( 400 22100 ) N ;
    - yellow + NET state[2] + DIRECTION OUTPUT + USE SIGNAL
      + PORT
        + LAYER met3 ( -400 -150 ) ( 400 150 )
        + PLACED ( 400 20740 ) N ;
END PINS
NETS 44 ;
    - _00_[0] ( _49_ D ) ( _39_ Y ) + USE SIGNAL
      + ROUTED met2 ( 23230 10030 ) ( * 11730 )
      NEW met1 ( 22770 11730 ) ( 23230 * )
      NEW li1 ( 23230 10030 ) L1M1_PR_MR
      NEW met1 ( 23230 10030 ) M1M2_PR
      NEW met1 ( 23230 11730 ) M1M2_PR
      NEW li1 ( 22770 11730 ) L1M1_PR_MR ;
    - _00_[1] ( _50_ D ) ( _38_ Y ) + USE SIGNAL
      + ROUTED met1 ( 25530 14110 ) ( 26450 * )
      NEW met2 ( 26450 12410 ) ( * 14110 )
      NEW li1 ( 26450 12410 ) L1M1_PR_MR
      NEW met1 ( 26450 12410 ) M1M2_PR
      NEW met1 ( 26450 14110 ) M1M2_PR
      NEW li1 ( 25530 14110 ) L1M1_PR_MR ;
    - _00_[2] ( _51_ D ) ( _36_ Y ) + USE SIGNAL
      + ROUTED met1 ( 25990 18530 ) ( 26450 * )
      NEW met2 ( 26450 18530 ) ( * 22610 )
      NEW li1 ( 25990 18530 ) L1M1_PR_MR
      NEW met1 ( 26450 18530 ) M1M2_PR
      NEW li1 ( 26450 22610 ) L1M1_PR_MR
      NEW met1 ( 26450 22610 ) M1M2_PR ;
    - _00_[3] ( _52_ D ) ( _27_ X ) + USE SIGNAL
      + ROUTED met1 ( 27830 30770 ) ( 29670 * )
      NEW met2 ( 27830 30770 ) ( * 33490 )
      NEW met1 ( 26450 33490 ) ( 27830 * )
      NEW li1 ( 29670 30770 ) L1M1_PR_MR
      NEW met1 ( 27830 30770 ) M1M2_PR
      NEW met1 ( 27830 33490 ) M1M2_PR
      NEW li1 ( 26450 33490 ) L1M1_PR_MR ;
    - _00_[4] ( _53_ D ) ( _26_ Y ) + USE SIGNAL
      + ROUTED met2 ( 26450 28730 ) ( * 30430 )
      NEW met1 ( 25070 30430 ) ( 26450 * )
      NEW li1 ( 26450 28730 ) L1M1_PR_MR
      NEW met1 ( 26450 28730 ) M1M2_PR
      NEW met1 ( 26450 30430 ) M1M2_PR
      NEW li1 ( 25070 30430 ) L1M1_PR_MR ;
    - _00_[5] ( _54_ D ) ( _25_ X ) + USE SIGNAL
      + ROUTED met2 ( 10810 31110 ) ( * 33490 )
      NEW met1 ( 9430 33490 ) ( 10810 * )
      NEW li1 ( 10810 31110 ) L1M1_PR_MR
      NEW met1 ( 10810 31110 ) M1M2_PR
      NEW met1 ( 10810 33490 ) M1M2_PR
      NEW li1 ( 9430 33490 ) L1M1_PR_MR ;
    - _00_[6] ( _55_ D ) ( _24_ Y ) + USE SIGNAL
      + ROUTED met1 ( 9890 26350 ) ( * 26690 )
      NEW met2 ( 14030 26690 ) ( * 30770 )
      NEW met1 ( 9890 26690 ) ( 14030 * )
      NEW li1 ( 9890 26350 ) L1M1_PR_MR
      NEW met1 ( 14030 26690 ) M1M2_PR
      NEW li1 ( 14030 30770 ) L1M1_PR_MR
      NEW met1 ( 14030 30770 ) M1M2_PR ;
    - _00_[7] ( _56_ D ) ( _45_ Y ) + USE SIGNAL
      + ROUTED met1 ( 7590 28730 ) ( 9430 * )
      NEW met1 ( 7590 28730 ) ( * 30430 )
      NEW met1 ( 7590 30430 ) ( 16790 * )
      NEW li1 ( 9430 28730 ) L1M1_PR_MR
      NEW li1 ( 16790 30430 ) L1M1_PR_MR ;
    - _01_ ( _46_ D ) ( _40_ X ) + USE SIGNAL
      + ROUTED met2 ( 7130 9690 ) ( * 14110 )
      NEW met1 ( 7130 14110 ) ( 8970 * )
      NEW li1 ( 7130 9690 ) L1M1_PR_MR
      NEW met1 ( 7130 9690 ) M1M2_PR
      NEW met1 ( 7130 14110 ) M1M2_PR
      NEW li1 ( 8970 14110 ) L1M1_PR_MR ;
    - _02_ ( _47_ D ) ( _42_ Y ) + USE SIGNAL
      + ROUTED met2 ( 6210 12410 ) ( * 14450 )
      NEW met1 ( 6210 14450 ) ( 13570 * )
      NEW li1 ( 6210 12410 ) L1M1_PR_MR
      NEW met1 ( 6210 12410 ) M1M2_PR
      NEW met1 ( 6210 14450 ) M1M2_PR
      NEW li1 ( 13570 14450 ) L1M1_PR_MR ;
    - _03_ ( _48_ D ) ( _44_ X ) + USE SIGNAL
      + ROUTED met2 ( 12650 18530 ) ( * 20230 )
      NEW met1 ( 10350 20230 ) ( 12650 * )
      NEW li1 ( 12650 18530 ) L1M1_PR_MR
      NEW met1 ( 12650 18530 ) M1M2_PR
      NEW met1 ( 12650 20230 ) M1M2_PR
      NEW li1 ( 10350 20230 ) L1M1_PR_MR ;
    - _04_ ( _36_ A ) ( _27_ B ) ( _21_ C ) ( _20_ B ) ( _19_ X ) + USE SIGNAL
      + ROUTED met1 ( 27830 26010 ) ( 28290 * )
      NEW met1 ( 27370 17850 ) ( 28290 * )
      NEW met2 ( 28290 17850 ) ( * 26010 )
      NEW met1 ( 28290 18190 ) ( 31510 * )
      NEW met1 ( 28290 17850 ) ( * 18190 )
      NEW met2 ( 28290 26010 ) ( * 27600 )
      NEW met2 ( 27370 31110 ) ( * 32130 )
      NEW met1 ( 22770 32130 ) ( 27370 * )
      NEW met1 ( 22770 31790 ) ( * 32130 )
      NEW met1 ( 21550 31790 ) ( 22770 * )
      NEW met2 ( 27370 27600 ) ( 28290 * )
      NEW met2 ( 27370 27600 ) ( * 31110 )
      NEW li1 ( 27830 26010 ) L1M1_PR_MR
      NEW met1 ( 28290 26010 ) M1M2_PR
      NEW li1 ( 27370 17850 ) L1M1_PR_MR
      NEW met1 ( 28290 17850 ) M1M2_PR
      NEW li1 ( 31510 18190 ) L1M1_PR_MR
      NEW li1 ( 27370 31110 ) L1M1_PR_MR
      NEW met1 ( 27370 31110 ) M1M2_PR
      NEW met1 ( 27370 32130 ) M1M2_PR
      NEW li1 ( 21550 31790 ) L1M1_PR_MR ;
    - _05_ ( _26_ B ) ( _20_ Y ) + USE SIGNAL
      + ROUTED met1 ( 28750 26690 ) ( 29210 * )
      NEW met1 ( 24150 31110 ) ( 26910 * )
      NEW met1 ( 26910 30430 ) ( * 31110 )
      NEW met1 ( 26910 30430 ) ( 29210 * )
      NEW met2 ( 29210 26690 ) ( * 30430 )
      NEW li1 ( 28750 26690 ) L1M1_PR_MR
      NEW met1 ( 29210 26690 ) M1M2_PR
      NEW li1 ( 24150 31110 ) L1M1_PR_MR
      NEW met1 ( 29210 30430 ) M1M2_PR ;
    - _06_ ( _25_ B ) ( _23_ C ) ( _22_ B ) ( _21_ X ) + USE SIGNAL
      + ROUTED met1 ( 9430 31110 ) ( * 31790 )
      NEW met2 ( 13110 31790 ) ( * 36890 )
      NEW met1 ( 17710 31790 ) ( * 32130 )
      NEW met1 ( 17710 32130 ) ( 22310 * )
      NEW met1 ( 9430 31790 ) ( 17710 * )
      NEW li1 ( 9430 31110 ) L1M1_PR_MR
      NEW met1 ( 13110 31790 ) M1M2_PR
      NEW li1 ( 13110 36890 ) L1M1_PR_MR
      NEW met1 ( 13110 36890 ) M1M2_PR
      NEW li1 ( 17710 31790 ) L1M1_PR_MR
      NEW li1 ( 22310 32130 ) L1M1_PR_MR ;
    - _07_ ( _24_ B ) ( _22_ Y ) + USE SIGNAL
      + ROUTED met1 ( 12650 35870 ) ( 13570 * )
      NEW met2 ( 12650 31110 ) ( * 35870 )
      NEW li1 ( 12650 31110 ) L1M1_PR_MR
      NEW met1 ( 12650 31110 ) M1M2_PR
      NEW met1 ( 12650 35870 ) M1M2_PR
      NEW li1 ( 13570 35870 ) L1M1_PR_MR ;
    - _08_ ( _45_ B ) ( _23_ Y ) + USE SIGNAL
      + ROUTED met1 ( 14950 31110 ) ( 18170 * )
      NEW met1 ( 18170 30770 ) ( * 31110 )
      NEW li1 ( 14950 31110 ) L1M1_PR_MR
      NEW li1 ( 18170 30770 ) L1M1_PR_MR ;
    - _09_ ( _33_ B ) ( _31_ B ) ( _28_ Y ) + USE SIGNAL
      + ROUTED met1 ( 20930 16830 ) ( * 17850 )
      NEW met1 ( 20930 17850 ) ( 24610 * )
      NEW met1 ( 24610 17850 ) ( * 19550 )
      NEW met1 ( 24610 19550 ) ( 26450 * )
      NEW met1 ( 18630 16830 ) ( * 17170 )
      NEW met1 ( 18630 17170 ) ( 20930 * )
      NEW li1 ( 20930 16830 ) L1M1_PR_MR
      NEW li1 ( 26450 19550 ) L1M1_PR_MR
      NEW li1 ( 18630 16830 ) L1M1_PR_MR ;
    - _10_ ( _33_ C ) ( _31_ C ) ( _29_ Y ) + USE SIGNAL
      + ROUTED met2 ( 18170 17510 ) ( * 27710 )
      NEW met1 ( 20470 17510 ) ( * 17850 )
      NEW met1 ( 18170 17850 ) ( 20470 * )
      NEW met1 ( 18170 17510 ) ( * 17850 )
      NEW li1 ( 18170 17510 ) L1M1_PR_MR
      NEW met1 ( 18170 17510 ) M1M2_PR
      NEW li1 ( 18170 27710 ) L1M1_PR_MR
      NEW met1 ( 18170 27710 ) M1M2_PR
      NEW li1 ( 20470 17510 ) L1M1_PR_MR ;
    - _11_ ( _31_ D ) ( _30_ X ) + USE SIGNAL
      + ROUTED met1 ( 17710 17510 ) ( * 18190 )
      NEW met1 ( 17710 18190 ) ( 24150 * )
      NEW li1 ( 17710 17510 ) L1M1_PR_MR
      NEW li1 ( 24150 18190 ) L1M1_PR_MR ;
    - _12_ ( _44_ A2 ) ( _42_ A ) ( _34_ A ) ( _31_ Y ) + USE SIGNAL
      + ROUTED met1 ( 16330 15130 ) ( 19090 * )
      NEW met2 ( 19090 15130 ) ( * 16830 )
      NEW met1 ( 14030 15130 ) ( * 15470 )
      NEW met1 ( 14030 15470 ) ( 16330 * )
      NEW met1 ( 16330 15130 ) ( * 15470 )
      NEW met1 ( 14490 15470 ) ( * 17510 )
      NEW met1 ( 11730 17510 ) ( 14490 * )
      NEW li1 ( 11730 17510 ) L1M1_PR_MR
      NEW li1 ( 16330 15130 ) L1M1_PR_MR
      NEW met1 ( 19090 15130 ) M1M2_PR
      NEW li1 ( 19090 16830 ) L1M1_PR_MR
      NEW met1 ( 19090 16830 ) M1M2_PR
      NEW li1 ( 14030 15130 ) L1M1_PR_MR ;
    - _13_ ( _33_ D ) ( _32_ Y ) + USE SIGNAL
      + ROUTED met2 ( 20010 17510 ) ( * 19550 )
      NEW li1 ( 20010 17510 ) L1M1_PR_MR
      NEW met1 ( 20010 17510 ) M1M2_PR
      NEW li1 ( 20010 19550 ) L1M1_PR_MR
      NEW met1 ( 20010 19550 ) M1M2_PR ;
    - _14_ ( _41_ B ) ( _39_ B_N ) ( _34_ B ) ( _33_ Y ) + USE SIGNAL
      + ROUTED met1 ( 20930 12070 ) ( 21390 * )
      NEW met1 ( 15870 12070 ) ( * 12410 )
      NEW met1 ( 15870 12410 ) ( 20930 * )
      NEW met1 ( 20930 12070 ) ( * 12410 )
      NEW met1 ( 15410 14790 ) ( * 15130 )
      NEW met1 ( 15410 14790 ) ( 21390 * )
      NEW met2 ( 21390 12070 ) ( * 16830 )
      NEW li1 ( 20930 12070 ) L1M1_PR_MR
      NEW met1 ( 21390 12070 ) M1M2_PR
      NEW li1 ( 15870 12070 ) L1M1_PR_MR
      NEW li1 ( 21390 16830 ) L1M1_PR_MR
      NEW met1 ( 21390 16830 ) M1M2_PR
      NEW li1 ( 15410 15130 ) L1M1_PR_MR
      NEW met1 ( 21390 14790 ) M1M2_PR ;
    - _15_ ( _44_ B1 ) ( _40_ S ) ( _38_ A ) ( _36_ B ) ( _34_ Y ) + USE SIGNAL
      + ROUTED met1 ( 10810 17170 ) ( 12190 * )
      NEW met2 ( 12190 14790 ) ( * 17170 )
      NEW met2 ( 26450 15130 ) ( * 17510 )
      NEW met1 ( 16330 15810 ) ( 26450 * )
      NEW met1 ( 14030 14450 ) ( * 14790 )
      NEW met1 ( 14030 14450 ) ( 15870 * )
      NEW met1 ( 12190 14790 ) ( 14030 * )
      NEW li1 ( 12190 14790 ) L1M1_PR_MR
      NEW li1 ( 10810 17170 ) L1M1_PR_MR
      NEW met1 ( 12190 17170 ) M1M2_PR
      NEW met1 ( 12190 14790 ) M1M2_PR
      NEW li1 ( 26450 15130 ) L1M1_PR_MR
      NEW met1 ( 26450 15130 ) M1M2_PR
      NEW li1 ( 26450 17510 ) L1M1_PR_MR
      NEW met1 ( 26450 17510 ) M1M2_PR
      NEW li1 ( 16330 15810 ) L1M1_PR_MR
      NEW met1 ( 26450 15810 ) M1M2_PR
      NEW li1 ( 15870 14450 ) L1M1_PR_MR ;
    - _16_ ( _36_ C ) ( _35_ Y ) + USE SIGNAL
      + ROUTED met1 ( 25530 17510 ) ( 25990 * )
      NEW met1 ( 25530 17510 ) ( * 18190 )
      NEW met1 ( 25530 18190 ) ( 27830 * )
      NEW li1 ( 25990 17510 ) L1M1_PR_MR
      NEW li1 ( 27830 18190 ) L1M1_PR_MR ;
    - _17_ ( _38_ B ) ( _37_ Y ) + USE SIGNAL
      + ROUTED met1 ( 25530 14450 ) ( * 15130 )
      NEW met1 ( 25530 14450 ) ( 29670 * )
      NEW li1 ( 25530 15130 ) L1M1_PR_MR
      NEW li1 ( 29670 14450 ) L1M1_PR_MR ;
    - _18_ ( _42_ B ) ( _41_ Y ) + USE SIGNAL
      + ROUTED met1 ( 13110 15130 ) ( 13570 * )
      NEW met2 ( 13570 15130 ) ( * 15300 )
      NEW met1 ( 14030 13090 ) ( 16330 * )
      NEW met2 ( 13570 15300 ) ( 14030 * )
      NEW met2 ( 14030 13090 ) ( * 15300 )
      NEW li1 ( 13110 15130 ) L1M1_PR_MR
      NEW met1 ( 13570 15130 ) M1M2_PR
      NEW met1 ( 14030 13090 ) M1M2_PR
      NEW li1 ( 16330 13090 ) L1M1_PR_MR ;
    - clk ( PIN clk ) ( clkbuf_0_clk A ) + USE CLOCK + NONDEFAULTRULE CTS_NDR_0
      + ROUTED met2 ( 14950 18020 ) ( * 22950 )
      NEW met3 ( 460 18020 0 ) ( 14950 * )
      NEW met2 ( 14950 18020 ) M2M3_PR
      NEW li1 TAPER ( 14950 22950 ) L1M1_PR_MR
      NEW met1 TAPER ( 14950 22950 ) M1M2_PR ;
    - clknet_0_clk ( clkbuf_1_1__f_clk A ) ( clkbuf_1_0__f_clk A ) ( clkbuf_0_clk X ) + USE CLOCK + NONDEFAULTRULE CTS_NDR_0
      + ROUTED met1 TAPER ( 17710 11730 ) ( 18170 * )
      NEW met2 ( 17710 11730 ) ( * 11900 )
      NEW met2 ( 17250 11900 ) ( 17710 * )
      NEW met1 TAPER ( 16330 23630 ) ( 17250 * )
      NEW met2 ( 17250 23630 ) ( * 33490 )
      NEW met1 TAPER ( 17250 33490 ) ( 18170 * )
      NEW met2 ( 17250 11900 ) ( * 23630 )
      NEW li1 TAPER ( 18170 11730 ) L1M1_PR_MR
      NEW met1 TAPER ( 17710 11730 ) M1M2_PR
      NEW li1 TAPER ( 16330 23630 ) L1M1_PR_MR
      NEW met1 TAPER ( 17250 23630 ) M1M2_PR
      NEW met1 TAPER ( 17250 33490 ) M1M2_PR
      NEW li1 TAPER ( 18170 33490 ) L1M1_PR_MR ;
    - clknet_1_0__leaf_clk ( clkload0 A ) ( _46_ CLK ) ( _47_ CLK ) ( _48_ CLK ) ( _49_ CLK ) ( _50_ CLK ) ( clkbuf_1_0__f_clk X ) + USE CLOCK
      + ROUTED met1 ( 4830 12410 ) ( * 13090 )
      NEW met1 ( 4830 13090 ) ( 8970 * )
      NEW met2 ( 8970 13090 ) ( * 20230 )
      NEW met1 ( 5290 9350 ) ( 8050 * )
      NEW met2 ( 8050 9350 ) ( * 13090 )
      NEW met1 ( 15410 9350 ) ( * 9690 )
      NEW met1 ( 8050 9350 ) ( 15410 * )
      NEW met1 ( 15410 9690 ) ( 21850 * )
      NEW met1 ( 19550 11390 ) ( 20470 * )
      NEW met1 ( 20470 9690 ) ( * 11390 )
      NEW met1 ( 25070 11390 ) ( * 12070 )
      NEW met1 ( 20470 11390 ) ( 25070 * )
      NEW li1 ( 4830 12410 ) L1M1_PR_MR
      NEW met1 ( 8970 13090 ) M1M2_PR
      NEW li1 ( 8970 20230 ) L1M1_PR_MR
      NEW met1 ( 8970 20230 ) M1M2_PR
      NEW li1 ( 5290 9350 ) L1M1_PR_MR
      NEW met1 ( 8050 9350 ) M1M2_PR
      NEW met1 ( 8050 13090 ) M1M2_PR
      NEW li1 ( 15410 9690 ) L1M1_PR_MR
      NEW li1 ( 21850 9690 ) L1M1_PR_MR
      NEW li1 ( 19550 11390 ) L1M1_PR_MR
      NEW li1 ( 25070 12070 ) L1M1_PR_MR ;
    - clknet_1_1__leaf_clk ( _51_ CLK ) ( _52_ CLK ) ( _53_ CLK ) ( _54_ CLK ) ( _55_ CLK ) ( _56_ CLK ) ( clkbuf_1_1__f_clk X ) + USE CLOCK
      + ROUTED met2 ( 8050 28390 ) ( * 34170 )
      NEW met1 ( 8050 26010 ) ( 8510 * )
      NEW met2 ( 8050 26010 ) ( * 28390 )
      NEW met1 ( 19550 34170 ) ( 25070 * )
      NEW met2 ( 25070 28730 ) ( * 34170 )
      NEW met2 ( 25070 23290 ) ( * 28730 )
      NEW met1 ( 8050 34170 ) ( 19550 * )
      NEW li1 ( 8050 34170 ) L1M1_PR_MR
      NEW li1 ( 8050 28390 ) L1M1_PR_MR
      NEW met1 ( 8050 28390 ) M1M2_PR
      NEW met1 ( 8050 34170 ) M1M2_PR
      NEW li1 ( 8510 26010 ) L1M1_PR_MR
      NEW met1 ( 8050 26010 ) M1M2_PR
      NEW li1 ( 19550 34170 ) L1M1_PR_MR
      NEW li1 ( 25070 34170 ) L1M1_PR_MR
      NEW li1 ( 25070 28730 ) L1M1_PR_MR
      NEW met1 ( 25070 28730 ) M1M2_PR
      NEW met1 ( 25070 34170 ) M1M2_PR
      NEW li1 ( 25070 23290 ) L1M1_PR_MR
      NEW met1 ( 25070 23290 ) M1M2_PR ;
    - cnt[0] ( _49_ Q ) ( _39_ A ) ( _37_ B ) ( _35_ A2 ) ( _32_ A ) ( _31_ A ) ( _19_ B ) + USE SIGNAL
      + ROUTED met1 ( 21850 12070 ) ( * 13090 )
      NEW met1 ( 21850 13090 ) ( 30130 * )
      NEW met2 ( 30130 13090 ) ( * 13260 )
      NEW met2 ( 30130 13260 ) ( 30590 * )
      NEW met1 ( 19550 13090 ) ( 21850 * )
      NEW met1 ( 28290 14790 ) ( 30590 * )
      NEW met1 ( 29210 17510 ) ( 30590 * )
      NEW met2 ( 30590 14790 ) ( * 17510 )
      NEW met2 ( 30590 17510 ) ( * 18530 )
      NEW met1 ( 20930 20230 ) ( * 20570 )
      NEW met1 ( 19550 20230 ) ( 20930 * )
      NEW met2 ( 19550 17510 ) ( * 20230 )
      NEW met2 ( 19550 13090 ) ( * 17510 )
      NEW met2 ( 30590 10370 ) ( * 14790 )
      NEW li1 ( 30590 10370 ) L1M1_PR_MR
      NEW met1 ( 30590 10370 ) M1M2_PR
      NEW li1 ( 21850 12070 ) L1M1_PR_MR
      NEW met1 ( 30130 13090 ) M1M2_PR
      NEW met1 ( 19550 13090 ) M1M2_PR
      NEW li1 ( 28290 14790 ) L1M1_PR_MR
      NEW met1 ( 30590 14790 ) M1M2_PR
      NEW li1 ( 29210 17510 ) L1M1_PR_MR
      NEW met1 ( 30590 17510 ) M1M2_PR
      NEW li1 ( 30590 18530 ) L1M1_PR_MR
      NEW met1 ( 30590 18530 ) M1M2_PR
      NEW li1 ( 19550 17510 ) L1M1_PR_MR
      NEW met1 ( 19550 17510 ) M1M2_PR
      NEW li1 ( 20930 20570 ) L1M1_PR_MR
      NEW met1 ( 19550 20230 ) M1M2_PR ;
    - cnt[1] ( _50_ Q ) ( _37_ A ) ( _35_ A1 ) ( _28_ A ) ( _19_ A ) + USE SIGNAL
      + ROUTED met1 ( 27830 15130 ) ( 33810 * )
      NEW met2 ( 28750 15130 ) ( * 16830 )
      NEW met1 ( 29670 16830 ) ( * 17170 )
      NEW met1 ( 28750 16830 ) ( 29670 * )
      NEW met1 ( 27370 20570 ) ( 28750 * )
      NEW met2 ( 28750 16830 ) ( * 20570 )
      NEW met2 ( 33810 13090 ) ( * 15130 )
      NEW li1 ( 33810 13090 ) L1M1_PR_MR
      NEW met1 ( 33810 13090 ) M1M2_PR
      NEW li1 ( 27830 15130 ) L1M1_PR_MR
      NEW met1 ( 33810 15130 ) M1M2_PR
      NEW li1 ( 28750 16830 ) L1M1_PR_MR
      NEW met1 ( 28750 16830 ) M1M2_PR
      NEW met1 ( 28750 15130 ) M1M2_PR
      NEW li1 ( 29670 17170 ) L1M1_PR_MR
      NEW li1 ( 27370 20570 ) L1M1_PR_MR
      NEW met1 ( 28750 20570 ) M1M2_PR ;
    - cnt[2] ( _51_ Q ) ( _35_ B1 ) ( _33_ A ) ( _30_ SLEEP ) ( _19_ C ) + USE SIGNAL
      + ROUTED met2 ( 27830 17510 ) ( * 22270 )
      NEW met1 ( 27830 22270 ) ( 33810 * )
      NEW met1 ( 30590 16830 ) ( 30670 * )
      NEW met1 ( 30590 16830 ) ( * 17170 )
      NEW met1 ( 30590 17170 ) ( 31050 * )
      NEW met1 ( 31050 17170 ) ( * 17850 )
      NEW met1 ( 28750 17850 ) ( 31050 * )
      NEW met1 ( 28750 17510 ) ( * 17850 )
      NEW met1 ( 27830 17510 ) ( 28750 * )
      NEW met1 ( 23230 17170 ) ( * 17510 )
      NEW met1 ( 23230 17170 ) ( 26910 * )
      NEW met1 ( 26910 17170 ) ( * 17510 )
      NEW met1 ( 26910 17510 ) ( 27830 * )
      NEW met1 ( 21850 17510 ) ( 23230 * )
      NEW li1 ( 27830 17510 ) L1M1_PR_MR
      NEW met1 ( 27830 17510 ) M1M2_PR
      NEW met1 ( 27830 22270 ) M1M2_PR
      NEW li1 ( 33810 22270 ) L1M1_PR_MR
      NEW li1 ( 30670 16830 ) L1M1_PR_MR
      NEW li1 ( 23230 17510 ) L1M1_PR_MR
      NEW li1 ( 21850 17510 ) L1M1_PR_MR ;
    - cnt[3] ( _52_ Q ) ( _28_ B ) ( _27_ A ) ( _21_ A ) ( _20_ A ) + USE SIGNAL
      + ROUTED met1 ( 28750 26010 ) ( 29670 * )
      NEW met1 ( 26450 20230 ) ( * 20570 )
      NEW met1 ( 26450 20230 ) ( 29670 * )
      NEW met2 ( 29670 20230 ) ( * 26010 )
      NEW met1 ( 27830 31450 ) ( 33810 * )
      NEW met2 ( 33810 31450 ) ( * 33150 )
      NEW met1 ( 20470 31110 ) ( * 31790 )
      NEW met1 ( 20470 31110 ) ( 22770 * )
      NEW met1 ( 22770 31110 ) ( * 31450 )
      NEW met1 ( 22770 31450 ) ( 23230 * )
      NEW met1 ( 23230 31450 ) ( * 31790 )
      NEW met1 ( 23230 31790 ) ( 27830 * )
      NEW met1 ( 27830 31450 ) ( * 31790 )
      NEW met2 ( 29670 26010 ) ( * 31450 )
      NEW li1 ( 28750 26010 ) L1M1_PR_MR
      NEW met1 ( 29670 26010 ) M1M2_PR
      NEW li1 ( 26450 20570 ) L1M1_PR_MR
      NEW met1 ( 29670 20230 ) M1M2_PR
      NEW li1 ( 27830 31450 ) L1M1_PR_MR
      NEW met1 ( 33810 31450 ) M1M2_PR
      NEW li1 ( 33810 33150 ) L1M1_PR_MR
      NEW met1 ( 33810 33150 ) M1M2_PR
      NEW met1 ( 29670 31450 ) M1M2_PR
      NEW li1 ( 20470 31790 ) L1M1_PR_MR ;
    - cnt[4] ( _53_ Q ) ( _29_ A ) ( _26_ A ) ( _21_ B ) + USE SIGNAL
      + ROUTED met2 ( 23690 29410 ) ( * 31450 )
      NEW met1 ( 23690 29410 ) ( 33810 * )
      NEW met1 ( 21390 30430 ) ( 23690 * )
      NEW met1 ( 19550 28730 ) ( * 29410 )
      NEW met1 ( 19550 29410 ) ( 23690 * )
      NEW li1 ( 23690 31450 ) L1M1_PR_MR
      NEW met1 ( 23690 31450 ) M1M2_PR
      NEW met1 ( 23690 29410 ) M1M2_PR
      NEW li1 ( 33810 29410 ) L1M1_PR_MR
      NEW li1 ( 21390 30430 ) L1M1_PR_MR
      NEW met1 ( 23690 30430 ) M1M2_PR
      NEW li1 ( 19550 28730 ) L1M1_PR_MR ;
    - cnt[5] ( _54_ Q ) ( _29_ B ) ( _25_ A ) ( _23_ A ) ( _22_ A ) + USE SIGNAL
      + ROUTED met1 ( 8970 31450 ) ( * 32130 )
      NEW met2 ( 16790 34850 ) ( * 36890 )
      NEW met1 ( 14030 36890 ) ( 16790 * )
      NEW met2 ( 16330 32130 ) ( * 34340 )
      NEW met2 ( 16330 34340 ) ( 16790 * )
      NEW met2 ( 16790 34340 ) ( * 34850 )
      NEW met1 ( 16330 29070 ) ( 19090 * )
      NEW met2 ( 16330 29070 ) ( * 32130 )
      NEW met1 ( 19080 31450 ) ( 19090 * )
      NEW met2 ( 19090 29070 ) ( * 31450 )
      NEW met1 ( 8970 32130 ) ( 16330 * )
      NEW li1 ( 8970 31450 ) L1M1_PR_MR
      NEW li1 ( 16790 34850 ) L1M1_PR_MR
      NEW met1 ( 16790 34850 ) M1M2_PR
      NEW met1 ( 16790 36890 ) M1M2_PR
      NEW li1 ( 14030 36890 ) L1M1_PR_MR
      NEW met1 ( 16330 32130 ) M1M2_PR
      NEW li1 ( 19090 29070 ) L1M1_PR_MR
      NEW met1 ( 16330 29070 ) M1M2_PR
      NEW li1 ( 19080 31450 ) L1M1_PR_MR
      NEW met1 ( 19090 31450 ) M1M2_PR
      NEW met1 ( 19090 29070 ) M1M2_PR
      NEW met1 ( 19090 31450 ) RECT ( 0 -70 345 70 )  ;
    - cnt[6] ( _55_ Q ) ( _29_ C ) ( _24_ A ) ( _23_ B ) + USE SIGNAL
      + ROUTED met2 ( 18630 31450 ) ( * 31620 )
      NEW met3 ( 14030 31620 ) ( 18630 * )
      NEW met2 ( 14030 31450 ) ( * 31620 )
      NEW met2 ( 18630 29410 ) ( * 31450 )
      NEW met1 ( 17250 26690 ) ( 18630 * )
      NEW met2 ( 18630 26690 ) ( * 29410 )
      NEW met1 ( 12190 31450 ) ( 14030 * )
      NEW li1 ( 12190 31450 ) L1M1_PR_MR
      NEW li1 ( 18630 31450 ) L1M1_PR_MR
      NEW met1 ( 18630 31450 ) M1M2_PR
      NEW met2 ( 18630 31620 ) M2M3_PR
      NEW met2 ( 14030 31620 ) M2M3_PR
      NEW met1 ( 14030 31450 ) M1M2_PR
      NEW li1 ( 18630 29410 ) L1M1_PR_MR
      NEW met1 ( 18630 29410 ) M1M2_PR
      NEW li1 ( 17250 26690 ) L1M1_PR_MR
      NEW met1 ( 18630 26690 ) M1M2_PR ;
    - cnt[7] ( _56_ Q ) ( _45_ A ) ( _29_ D ) + USE SIGNAL
      + ROUTED met1 ( 15410 29410 ) ( 16790 * )
      NEW met2 ( 15410 29410 ) ( * 31450 )
      NEW met1 ( 17710 28390 ) ( * 28730 )
      NEW met1 ( 16790 28730 ) ( 17710 * )
      NEW li1 ( 16790 29410 ) L1M1_PR_MR
      NEW met1 ( 15410 29410 ) M1M2_PR
      NEW li1 ( 15410 31450 ) L1M1_PR_MR
      NEW met1 ( 15410 31450 ) M1M2_PR
      NEW li1 ( 17710 28390 ) L1M1_PR_MR
      NEW li1 ( 16790 28730 ) L1M1_PR_MR ;
    - limit[0] + USE SIGNAL ;
    - red ( PIN red ) ( _43_ Y ) + USE SIGNAL
      + ROUTED met3 ( 460 19380 0 ) ( 1150 * )
      NEW met2 ( 1150 18530 ) ( * 19380 )
      NEW met1 ( 1150 18530 ) ( 8510 * )
      NEW met2 ( 1150 19380 ) M2M3_PR
      NEW met1 ( 1150 18530 ) M1M2_PR
      NEW li1 ( 8510 18530 ) L1M1_PR_MR ;
    - rst_n ( PIN rst_n ) ( _56_ RESET_B ) ( _55_ RESET_B ) ( _54_ RESET_B ) ( _53_ RESET_B ) ( _52_ RESET_B ) ( _51_ RESET_B )
      ( _50_ RESET_B ) ( _49_ RESET_B ) ( _48_ RESET_B ) ( _47_ RESET_B ) ( _46_ SET_B ) + USE SIGNAL
      + ROUTED met2 ( 28750 9690 ) ( * 9860 )
      NEW met2 ( 28750 9860 ) ( 29210 * )
      NEW met2 ( 29210 9860 ) ( * 11730 )
      NEW met2 ( 13570 20910 ) ( * 26350 )
      NEW met1 ( 12190 10030 0 ) ( 13110 * )
      NEW met2 ( 13110 10030 ) ( * 15980 )
      NEW met2 ( 13110 15980 ) ( 13570 * )
      NEW met2 ( 13570 15980 ) ( * 20910 )
      NEW met1 ( 12190 11730 0 ) ( 13110 * )
      NEW met3 ( 460 22100 0 ) ( 13570 * )
      NEW met2 ( 28750 28050 ) ( * 33490 )
      NEW met2 ( 28750 22610 ) ( * 28050 )
      NEW met2 ( 28750 21420 ) ( 29210 * )
      NEW met2 ( 28750 21420 ) ( * 22610 )
      NEW met1 ( 16330 20910 0 ) ( 29210 * )
      NEW met2 ( 14490 28050 ) ( * 33490 )
      NEW met2 ( 14490 26350 ) ( 14950 * )
      NEW met2 ( 14490 26350 ) ( * 28050 )
      NEW met2 ( 29210 11730 ) ( * 21420 )
      NEW met1 ( 29210 11730 ) M1M2_PR
      NEW met1 ( 28750 9690 ) M1M2_PR
      NEW met1 ( 13570 26350 ) M1M2_PR
      NEW met1 ( 13570 20910 ) M1M2_PR
      NEW met1 ( 13110 10030 ) M1M2_PR
      NEW met1 ( 13110 11730 ) M1M2_PR
      NEW met2 ( 13570 22100 ) M2M3_PR
      NEW met1 ( 28750 28050 ) M1M2_PR
      NEW met1 ( 28750 33490 ) M1M2_PR
      NEW met1 ( 28750 22610 ) M1M2_PR
      NEW met1 ( 29210 20910 ) M1M2_PR
      NEW met1 ( 14490 28050 ) M1M2_PR
      NEW met1 ( 14490 33490 ) M1M2_PR
      NEW met1 ( 14950 26350 ) M1M2_PR ;
    - state[0] ( PIN green ) ( _46_ Q ) ( _44_ B2 ) ( _43_ B ) ( _40_ A0 ) + USE SIGNAL
      + ROUTED met2 ( 8510 17510 ) ( * 23460 )
      NEW met3 ( 460 23460 0 ) ( 8510 * )
      NEW met1 ( 9890 17510 ) ( * 17850 )
      NEW met1 ( 8510 17850 ) ( 9890 * )
      NEW met1 ( 8510 17510 ) ( * 17850 )
      NEW met1 ( 8510 15810 ) ( 10810 * )
      NEW met2 ( 8510 15810 ) ( * 17510 )
      NEW met1 ( 14260 10370 ) ( 14490 * )
      NEW met2 ( 14030 15810 ) ( * 15980 )
      NEW met2 ( 14030 15980 ) ( 14490 * )
      NEW met1 ( 10810 15810 ) ( 14030 * )
      NEW met2 ( 14490 10370 ) ( * 15980 )
      NEW li1 ( 8510 17510 ) L1M1_PR_MR
      NEW met1 ( 8510 17510 ) M1M2_PR
      NEW met2 ( 8510 23460 ) M2M3_PR
      NEW li1 ( 9890 17510 ) L1M1_PR_MR
      NEW li1 ( 10810 15810 ) L1M1_PR_MR
      NEW met1 ( 8510 15810 ) M1M2_PR
      NEW li1 ( 14260 10370 ) L1M1_PR_MR
      NEW met1 ( 14490 10370 ) M1M2_PR
      NEW met1 ( 14030 15810 ) M1M2_PR ;
    - state[1] ( _47_ Q ) ( _41_ A ) ( _40_ A1 ) + USE SIGNAL
      + ROUTED met1 ( 11270 13090 ) ( 13570 * )
      NEW met2 ( 11270 13090 ) ( * 14790 )
      NEW met1 ( 16790 11730 ) ( * 12070 )
      NEW met1 ( 13570 11730 ) ( 16790 * )
      NEW met1 ( 13570 11390 ) ( * 11730 )
      NEW li1 ( 13570 13090 ) L1M1_PR_MR
      NEW met1 ( 11270 13090 ) M1M2_PR
      NEW li1 ( 11270 14790 ) L1M1_PR_MR
      NEW met1 ( 11270 14790 ) M1M2_PR
      NEW li1 ( 16790 12070 ) L1M1_PR_MR
      NEW li1 ( 13570 11390 ) L1M1_PR_MR ;
    - state[2] ( PIN yellow ) ( _48_ Q ) ( _44_ A1 ) ( _43_ A ) ( _32_ B ) ( _30_ A ) + USE SIGNAL
      + ROUTED met1 ( 9430 17170 ) ( * 17510 )
      NEW met1 ( 6670 17170 ) ( 9430 * )
      NEW met2 ( 6670 17170 ) ( * 20740 )
      NEW met3 ( 460 20740 0 ) ( 6670 * )
      NEW met1 ( 10350 17510 ) ( 11270 * )
      NEW met1 ( 10350 17170 ) ( * 17510 )
      NEW met1 ( 9430 17170 ) ( 10350 * )
      NEW met1 ( 19090 20570 ) ( 20010 * )
      NEW met1 ( 19090 19890 ) ( * 20570 )
      NEW met1 ( 19090 19890 ) ( 22310 * )
      NEW met2 ( 22310 17170 ) ( * 19890 )
      NEW met1 ( 17710 19890 ) ( 19090 * )
      NEW met1 ( 17710 19550 ) ( * 19890 )
      NEW met1 ( 6670 19550 ) ( 17710 * )
      NEW li1 ( 9430 17510 ) L1M1_PR_MR
      NEW met1 ( 6670 17170 ) M1M2_PR
      NEW met2 ( 6670 20740 ) M2M3_PR
      NEW li1 ( 11270 17510 ) L1M1_PR_MR
      NEW met1 ( 6670 19550 ) M1M2_PR
      NEW li1 ( 20010 20570 ) L1M1_PR_MR
      NEW met1 ( 22310 19890 ) M1M2_PR
      NEW li1 ( 22310 17170 ) L1M1_PR_MR
      NEW met1 ( 22310 17170 ) M1M2_PR
      NEW li1 ( 17710 19890 ) L1M1_PR_MR ;
END NETS
END DESIGN
'''
open('semaforo.def','w').write(DEF_TEXT)
DEF = 'semaforo.def'
print('Layout de demo:', DEF, '|', len(DEF_TEXT), 'chars')

## 2 - Leer el layout (el `.def`)

El **DEF** es el layout en texto: el area del chip (`DIEAREA`), donde quedo cada
celda (`COMPONENTS`) y como se cablearon las redes (`NETS`, con segmentos de
metal por capa). Lo parseamos a mano -- nada de magia, es leer texto.

In [ ]:
U = 1000.0  # DBU por micra (UNITS DISTANCE MICRONS 1000)
txt = open(DEF).read()

# --- area del chip ---
m = re.search(r'DIEAREA\s*\(\s*(-?\d+)\s+(-?\d+)\s*\)\s*\(\s*(-?\d+)\s+(-?\d+)\s*\)', txt)
dx0,dy0,dx1,dy1 = [int(v)/U for v in m.groups()]

# --- celdas colocadas ---
cblk = txt[txt.index('COMPONENTS'):txt.index('END COMPONENTS')]
comps = []
for mm in re.finditer(r'-\s+(\S+)\s+(\S+)\s+\+.*?\(\s*(-?\d+)\s+(-?\d+)\s*\)\s*(\w+)\s*;', cblk, re.S):
    name, cell, x, y, _ = mm.groups()
    comps.append((name, cell, int(x)/U, int(y)/U))

def kind(cell, name):
    if 'tapvpwrvgnd' in cell or name.startswith('TAP'):  return 'tap'
    if 'decap' in cell or name.startswith('PHY_EDGE'):   return 'endcap'
    if 'clkbuf' in cell or name.startswith('clk'):       return 'clk'
    if '__df' in cell:                                   return 'ff'
    return 'comb'

# --- pines de IO ---
pins = []
for mm in re.finditer(r'-\s+(\w+)\s+\+\s+NET.*?\+\s+PLACED\s*\(\s*(-?\d+)\s+(-?\d+)\s*\)', txt, re.S):
    pins.append((mm.group(1), int(mm.group(2))/U, int(mm.group(3))/U))

# --- routing: segmentos por red (y si la red es de reloj) ---
nblk = txt[txt.index('\nNETS'):txt.index('END NETS')]
LAYC = {'li1':'#9ca3af','met1':'#2563eb','met2':'#dc2626','met3':'#16a34a',
        'met4':'#d97706','met5':'#7c3aed'}
nets = {}          # nombre -> list[(layer,(x0,y0),(x1,y1))]
clock_nets = set()

def parse_route(route_txt):
    segs = []
    toks = route_txt.replace('(',' ( ').replace(')',' ) ').split()
    i, layer, cur, expect = 0, None, None, False
    while i < len(toks):
        t = toks[i]
        if t in ('ROUTED','NEW'): expect=True; cur=None; i+=1; continue
        if expect:
            layer=t; expect=False; i+=1
            if i<len(toks) and toks[i]=='TAPER': i+=1
            continue
        if t=='TAPER': i+=1; continue
        if t=='RECT': j=toks.index(')',i); i=j+1; continue
        if t=='(':
            j=toks.index(')',i); nums=toks[i+1:j]
            nx = cur[0] if nums[0]=='*' else int(nums[0])/U
            ny = cur[1] if nums[1]=='*' else int(nums[1])/U
            if cur is not None and layer in LAYC: segs.append((layer,cur,(nx,ny)))
            cur=(nx,ny); i=j+1; continue
        i+=1
    return segs

for rec in nblk.split(';'):
    k = rec.find('+ ROUTED')
    if k==-1: continue
    nm = rec.strip().split()[1]
    nets[nm] = parse_route(rec[k+8:])
    if 'USE CLOCK' in rec: clock_nets.add(nm)

nseg = sum(len(v) for v in nets.values())
print(f'die {dx1-dx0:.1f} x {dy1-dy0:.1f} um | celdas {len(comps)} | pines {len(pins)} | segmentos {nseg}')
print('redes de reloj:', sorted(clock_nets))

## 3 - El placement: donde quedo cada celda

Cada rectangulo es una celda, coloreada por tipo. Fijate en los **buffers de
reloj** (naranja) repartidos entre los **flip-flops** (azul): eso es el CTS
preparando el reparto del reloj. Las columnas grises son tapcells/endcaps que
**mete el PnR** (no estaban en el netlist).

In [ ]:
COLOR = {'comb':'#C8742B','ff':'#2563eb','clk':'#e8590c','tap':'#9ca3af','endcap':'#d1d5db'}
LABEL = {'comb':'combinacional','ff':'flip-flop','clk':'buffer de reloj (CTS)','tap':'tapcell','endcap':'endcap/decap'}
def ver_placement():
    fig, ax = plt.subplots(figsize=(7,7))
    ax.add_patch(Rectangle((dx0,dy0),dx1-dx0,dy1-dy0,fill=False,ec='black',lw=1.5))
    for name,cell,x,y in comps:
        k=kind(cell,name); w=2.0 if k in('ff','clk') else (1.4 if k=='comb' else 1.2)
        ax.add_patch(Rectangle((x,y),w,2.72,fc=COLOR[k],ec='white',lw=0.3))
    ax.set_xlim(dx0-2,dx1+2); ax.set_ylim(dy0-2,dy1+2); ax.set_aspect('equal')
    ax.set_title('semaforo - placement'); ax.set_xlabel('um'); ax.set_ylabel('um')
    ax.legend(handles=[Line2D([0],[0],marker='s',ls='',mfc=COLOR[k],mec='w',ms=10,label=LABEL[k]) for k in LABEL],
              loc='upper center', bbox_to_anchor=(0.5,-0.08), ncol=3, fontsize=8, frameon=False)
    plt.tight_layout(); plt.show()
ver_placement()

## 4 - El routing: las tres leches (reloj + senal)

Aqui estan los **cables reales** de metal. Usa el menu para **resaltar una red**
y ver su recorrido. Prueba una red de reloj (`clk`, `clknet_...`): veras el
**arbol de reloj** (el CTS) repartiendose hacia los flip-flops. El checkbox
deja ver *solo* el reloj -- las otras 'leches' (las senales de datos) se apagan.

In [ ]:
def ver_routing(resaltar='(todas)', solo_reloj=False):
    fig, ax = plt.subplots(figsize=(7.5,7.5))
    ax.add_patch(Rectangle((dx0,dy0),dx1-dx0,dy1-dy0,fill=False,ec='black',lw=1.5))
    for name,cell,x,y in comps:
        ax.add_patch(Rectangle((x,y),1.4,2.72,fc='#eceff1',ec='none'))
    for nm, segs in nets.items():
        if solo_reloj and nm not in clock_nets: continue
        hot = (resaltar==nm)
        for layer,p0,p1 in segs:
            if resaltar=='(todas)' or solo_reloj:
                ax.plot([p0[0],p1[0]],[p0[1],p1[1]],color=LAYC[layer],lw=1.3)
            elif hot:
                ax.plot([p0[0],p1[0]],[p0[1],p1[1]],color='#111111',lw=2.6,zorder=5)
            else:
                ax.plot([p0[0],p1[0]],[p0[1],p1[1]],color='#d8dde3',lw=0.8)
    for pn,px,py in pins:
        ax.plot(px,py,'ks',ms=5); ax.annotate(pn,(px,py),fontsize=7,xytext=(3,3),textcoords='offset points')
    ax.set_xlim(dx0-2,dx1+2); ax.set_ylim(dy0-2,dy1+2); ax.set_aspect('equal')
    ttl = f'routing -- resaltando: {resaltar}' if (resaltar!='(todas)' and not solo_reloj) else 'routing'
    ax.set_title(ttl); ax.set_xlabel('um'); ax.set_ylabel('um')
    if resaltar=='(todas)' or solo_reloj:
        ax.legend(handles=[Line2D([0],[0],color=LAYC[k],lw=3,label=k) for k in LAYC],
                  loc='upper center', bbox_to_anchor=(0.5,-0.08), ncol=6, fontsize=8, frameon=False)
    plt.tight_layout(); plt.show()

opts = ['(todas)'] + sorted(clock_nets) + sorted(n for n in nets if n not in clock_nets)
interact(ver_routing, resaltar=Dropdown(options=opts, value='(todas)', description='Resaltar'),
         solo_reloj=Checkbox(value=False, description='solo reloj'));

## 5 - Metricas del layout (la 'A' y la 'T' del PPA)

El PnR no solo coloca: **agrega celdas** (tapcells, endcaps, buffers de CTS) y,
con los cables reales, el timing deja de ser estimado.

In [ ]:
from collections import Counter
c = Counter(kind(cell,name) for name,cell,_,_ in comps)
print('Celdas por tipo (lo que hay en el layout):')
for k in ['comb','ff','clk','tap','endcap']:
    print(f'  {LABEL[k]:22} {c.get(k,0)}')
print(f'  {"TOTAL":22} {sum(c.values())}')
print()
print('Resultados del PnR (reporte de OpenROAD, CW=8):')
print('  area del core   ~ 489 um^2   (die 41.5 x 41.5 um, util ~43%)')
print('  slack (timing)  = +8.50 ns   sobre un reloj de 10 ns, 0 violaciones')
print()
print('Celdas que NO estaban en el netlist (las mete el PnR):')
print(f"  tapcells + endcaps = {c.get('tap',0)+c.get('endcap',0)}   |   buffers de reloj (CTS) = {c.get('clk',0)}")

## Lo que sigue

Ya tenemos el layout: celdas colocadas, reloj repartido (CTS) y todo cableado.
Con los **cables reales** (R y C), el timing ya no se estima: toca **firmarlo**
en el signoff estatico.

🍰 Pastel Tres Leches &nbsp;->&nbsp; 🍮 Cajeta (STA / Signoff)